# Train BPR

In [10]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

In [11]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import BPR
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [12]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "target" 
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

## Create dataset

In [13]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "category"] 
    },
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 1024 * 32,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": "full",
    },
    "metrics": ["Recall", "NDCG", "MRR"],
    "valid_metric": "NDCG@10",
    "seed": SEED,
}

config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [14]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
18 Jun 19:33    INFO  [Training]: train_batch_size = [1024] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
18 Jun 19:33    INFO  [Evaluation]: eval_batch_size = [32768] eval_args: [{'split': None, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'fu

## Train BPR

In [16]:
model: BPR = BPR(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

18 Jun 19:45    INFO  epoch 0 training [time: 0.61s, train loss: 54.7563]
18 Jun 19:45    INFO  epoch 0 evaluating [time: 13.35s, valid_score: 0.003000]
18 Jun 19:45    INFO  valid result: 
recall@10 : 0.0027    ndcg@10 : 0.003    mrr@10 : 0.0053
18 Jun 19:45    INFO  Saving current: saved/BPR-Jun-18-2026_19-45-41.pth
18 Jun 19:45    INFO  epoch 1 training [time: 0.56s, train loss: 54.4052]
18 Jun 19:46    INFO  epoch 1 evaluating [time: 12.31s, valid_score: 0.005300]
18 Jun 19:46    INFO  valid result: 
recall@10 : 0.0056    ndcg@10 : 0.0053    mrr@10 : 0.0081
18 Jun 19:46    INFO  Saving current: saved/BPR-Jun-18-2026_19-45-41.pth
18 Jun 19:46    INFO  epoch 2 training [time: 0.54s, train loss: 52.7078]
18 Jun 19:46    INFO  epoch 2 evaluating [time: 11.66s, valid_score: 0.005800]
18 Jun 19:46    INFO  valid result: 
recall@10 : 0.0063    ndcg@10 : 0.0058    mrr@10 : 0.0091
18 Jun 19:46    INFO  Saving current: saved/BPR-Jun-18-2026_19-45-41.pth
18 Jun 19:46    INFO  epoch 3 training

In [17]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0085
Best valid result:
  recall@10: 0.0078
  ndcg@10: 0.0085
  mrr@10: 0.0125


## Evaluate on test set

In [18]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

Test results (Overall):
  recall@10: 0.0052
  ndcg@10: 0.0047
  mrr@10: 0.0070


In [19]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    '''
    Evaluate the model on a subset of interactions defined by `mask`.
    '''
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold", tok["2"]: "new"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)

18 Jun 20:01    INFO  Loading model structure and parameters from saved/BPR-Jun-18-2026_19-45-41.pth



Evaluation (warm)
--------------------
  Interactions: 4495
  recall@10: 0.0080
  ndcg@10: 0.0082
  mrr@10: 0.0135


18 Jun 20:01    INFO  Loading model structure and parameters from saved/BPR-Jun-18-2026_19-45-41.pth



Evaluation (cold)
--------------------
  Interactions: 2678
  recall@10: 0.0042
  ndcg@10: 0.0036
  mrr@10: 0.0041


18 Jun 20:01    INFO  Loading model structure and parameters from saved/BPR-Jun-18-2026_19-45-41.pth



Evaluation (new)
--------------------
  Interactions: 2826
  recall@10: 0.0042
  ndcg@10: 0.0032
  mrr@10: 0.0053
